In [0]:
%run ./engine

In [0]:
%run ./omop_config

In [0]:
# ============================================================================
# The active dataset config + derived roles.
# ============================================================================
cfg = OMOP_CONFIG
PROFILER = cfg.profiler
RARE = cfg.rare
VALPROT_OVERRIDE = cfg.valprot_override

SRC = f"{cfg.source['catalog']}.{cfg.source['schema']}"
TGT = f"{cfg.target['catalog']}.{cfg.target['schema']}"
VOLUME = cfg.target["volume"]
GEN_DIR = f"{VOLUME}/generators"
GEN_REFERENCE = f"{GEN_DIR}/reference.zip"
GEN_CORE = f"{GEN_DIR}/core.zip"
CONFIG_DIR = f"{VOLUME}/config"
REPORT_DIR = f"{VOLUME}/reports"
FIXTURE_DIR = f"{VOLUME}/fixtures"
MANIFEST_PATH = f"{CONFIG_DIR}/run_manifest.json"
LEDGER_PATH = f"{CONFIG_DIR}/epsilon_ledger.json"
assert SRC == "4_prod.omop" and TGT == "8_dev.synth_omop", "cfg catalog drift — STOP"

SEED = cfg.extra["cohort"]["seed"]
COHORT = cfg.extra["cohort"]
PER_VISIT_KEEP = COHORT["per_visit_keep"]
NULL_FK_RATE = cfg.extra.get("null_fk_rate", {})
SRC_PERSONS = cfg.extra["src_persons"]
SPARK_DATE_MIN = cfg.extra["spark_date_min"]
SPARK_DATE_MAX = cfg.extra["spark_date_max"]
MAX_PANDAS_ROWS = cfg.extra["max_pandas_rows"]
MAX_PANDAS_GB = cfg.extra["max_pandas_gb"]
PK_ID_BASE = pk_id_base(cfg)
PK_BAND_SIZE = cfg.pk_band_size
GRAPH = cfg.tables
PII = cfg.pii

SUBJECT = cfg.subject
SUBJ_PK = cfg.tables[SUBJECT].pk
CORE = topo_order(cfg, "core")
REF_ORDER = topo_order(cfg, "reference")
CHAIN = list(cfg.reference_chain or [])
CHAIN_KEYS = chain_keys(cfg)
LEAF_KEY = CHAIN_KEYS[0] if CHAIN_KEYS else None
_KEY_TABLE = {cfg.tables[t].pk: t for t in CHAIN}

def _ctx_parent(t):
    fk = cfg.tables[t].context_fk
    return fk[1] if fk else None

HUBS = [t for t in CORE if _ctx_parent(t) == SUBJECT
        and any(_ctx_parent(c) == t for c in CORE)]
BASE_RATE_TABLES = [t for t in CORE if cfg.tables[t].base_rate_table]
SUBJECT_CHILDREN = [t for t in CORE if _ctx_parent(t) == SUBJECT
                    and t not in HUBS and t not in BASE_RATE_TABLES]
HUB_CHILDREN = {h: [t for t in CORE if _ctx_parent(t) == h] for h in HUBS}
SPLIT_GROUPS = {}
for _t in CORE:
    _sg = cfg.tables[_t].split_group
    if _sg:
        SPLIT_GROUPS.setdefault(_sg, []).append(_t)
CAP_TABLES = {t: cfg.tables[t].caps["per_parent"] for t in CORE
              if cfg.tables[t].caps and cfg.tables[t].caps.get("per_parent")}
VALPROT_OFF_VISIT = [t for h in HUBS for t in HUB_CHILDREN[h] if t in cfg.disable_valprot_tables]
ALL_FINAL_TABLES = topo_order(cfg) + list(SPLIT_GROUPS)

import pandas as pd
import numpy as np
import json, traceback, os, uuid, secrets
from datetime import datetime, timezone
from pyspark.sql import types as T
from pyspark.sql import functions as F

def read_manifest():
    try:
        with open(MANIFEST_PATH) as f:
            return json.load(f)
    except Exception:
        return None

def write_manifest(m):
    os.makedirs(CONFIG_DIR, exist_ok=True)
    with open(MANIFEST_PATH, "w") as f:
        json.dump(m, f, indent=2, default=str)

def _utcnow():
    return datetime.now(timezone.utc).isoformat()

In [0]:
dbutils.widgets.dropdown("mode", "full", ["full", "fixture"])
dbutils.widgets.dropdown("stage", "auto", ["auto", "pre", "post"])
dbutils.widgets.text("cohort_persons", str(COHORT["persons"]))
dbutils.widgets.text("meas_cap", str(COHORT["meas_cap"]))
dbutils.widgets.text("obs_cap", str(COHORT["obs_cap"]))
dbutils.widgets.dropdown("force_ignore_manifest", "false", ["false", "true"])
# R4b: release = non-invertible (OS-entropy noise seed, unpublished, no raw counts);
# deterministic = reproducible (cohort seed, published) — TEST/regression only, NOT a release.
dbutils.widgets.dropdown("profiler_seed_mode", "release", ["release", "deterministic"])

mode = dbutils.widgets.get("mode")
stage = dbutils.widgets.get("stage")
cohort_persons = int(dbutils.widgets.get("cohort_persons"))
meas_cap = int(dbutils.widgets.get("meas_cap"))
obs_cap = int(dbutils.widgets.get("obs_cap"))
force_ignore_manifest = dbutils.widgets.get("force_ignore_manifest") == "true"
profiler_seed_mode = dbutils.widgets.get("profiler_seed_mode")
if mode == "fixture":
    cohort_persons = 200
    meas_cap = 100
    obs_cap = 100
FRACTION_BPS = max(1, int(round(cohort_persons / SRC_PERSONS * 10_000)))
print(f"mode={mode} stage={stage} cohort_persons={cohort_persons} FRACTION_BPS={FRACTION_BPS} "
      f"meas_cap={meas_cap} obs_cap={obs_cap} per_visit_keep={PER_VISIT_KEEP} "
      f"force_ignore_manifest={force_ignore_manifest} profiler_seed_mode={profiler_seed_mode}")
print(f"per-parent caps ENFORCED on generated output: {CAP_TABLES}")
print(f"PII scrubbed (nulled + gated) per table: {PII}")

In [0]:
# =====================================================================================
# §0 PROVISION — idempotent schema + volume setup.
# =====================================================================================
def provision():
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {TGT} COMMENT 'High-fidelity synthetic data "
              f"(MOSTLY AI). See synth_omop/pipeline notebooks.'")
    spark.sql(f"CREATE VOLUME IF NOT EXISTS {TGT}.artifacts COMMENT 'Generators, QA reports, "
              f"config snapshots, fixtures.'")
    for sub in [GEN_DIR, REPORT_DIR, CONFIG_DIR, FIXTURE_DIR]:
        os.makedirs(sub, exist_ok=True)
    print("PROVISION COMPLETE:", os.listdir(VOLUME))

In [0]:
# =====================================================================================
# §1 COHORT SAMPLER — deterministic hash cohort, visit-aware caps, observation split,
#    Spark-side date pre-clamp. NOTE: config-#1-shaped groupings.
# =====================================================================================
def src_columns(table: str) -> list:
    return [f.name for f in spark.table(f"{SRC}.{table}").schema.fields]

def date_clamp_select(table: str, spec) -> str:
    """SELECT clause clamping every date_col into the Spark-side representable range."""
    cols = src_columns(table)
    dset = set(spec.date_cols)
    pieces = []
    for c in cols:
        if c in dset:
            pieces.append(
                f"CASE WHEN `{c}` < TIMESTAMP'{SPARK_DATE_MIN}' THEN TIMESTAMP'{SPARK_DATE_MIN}' "
                f"WHEN `{c}` > TIMESTAMP'{SPARK_DATE_MAX}' THEN TIMESTAMP'{SPARK_DATE_MAX}' "
                f"ELSE `{c}` END AS `{c}`"
            )
        else:
            pieces.append(f"`{c}`")
    return ",\n  ".join(pieces)


def build_cohort():
    """Writes {TGT}.cohort_<table> for every declared table. Returns n_person."""
    spark.sql(f"""
    CREATE OR REPLACE TABLE {TGT}.cohort_person AS
    SELECT
      {date_clamp_select('person', GRAPH['person'])}
    FROM {SRC}.person
    WHERE pmod(abs(hash(person_id)), 10000) < {FRACTION_BPS}
    """)
    n_person = spark.table(f"{TGT}.cohort_person").count()
    print(f"cohort_person rows = {n_person}")
    assert n_person > 0, "empty person cohort — increase cohort_persons"
    cohort_ids = f"(SELECT person_id FROM {TGT}.cohort_person)"

    spark.sql(f"""
    CREATE OR REPLACE TABLE {TGT}.cohort_visit_occurrence AS
    SELECT
      {date_clamp_select('visit_occurrence', GRAPH['visit_occurrence'])}
    FROM {SRC}.visit_occurrence
    WHERE person_id IN {cohort_ids}
    """)
    print(f"cohort_visit_occurrence rows = {spark.table(f'{TGT}.cohort_visit_occurrence').count()}")

    for t in ["observation_period", "death", "condition_era", "drug_era", "dose_era"]:
        spark.sql(f"""
        CREATE OR REPLACE TABLE {TGT}.cohort_{t} AS
        SELECT
          {date_clamp_select(t, GRAPH[t])}
        FROM {SRC}.{t}
        WHERE person_id IN {cohort_ids}
        """)
        print(f"cohort_{t} rows = {spark.table(f'{TGT}.cohort_{t}').count()}")

    spark.sql(f"""
    CREATE OR REPLACE TABLE {TGT}.cohort_observation_person AS
    SELECT
      {date_clamp_select('observation', GRAPH['observation_person'])}
    FROM {SRC}.observation
    WHERE person_id IN {cohort_ids} AND visit_occurrence_id IS NULL
    """)
    print(f"cohort_observation_person rows = {spark.table(f'{TGT}.cohort_observation_person').count()}")

    obs_v_select = date_clamp_select("observation", GRAPH["observation_visit"])
    spark.sql(f"""
    CREATE OR REPLACE TABLE {TGT}.cohort_observation_visit AS
    WITH base AS (
      SELECT {obs_v_select}
      FROM {SRC}.observation
      WHERE person_id IN {cohort_ids} AND visit_occurrence_id IS NOT NULL
    ),
    ranked AS (
      SELECT *,
        row_number() OVER (PARTITION BY person_id, visit_occurrence_id ORDER BY rand({SEED})) AS rn_visit,
        row_number() OVER (PARTITION BY person_id ORDER BY rand({SEED})) AS rn_person
      FROM base
    )
    SELECT * EXCEPT (rn_visit, rn_person) FROM ranked
    WHERE rn_visit <= {PER_VISIT_KEEP} AND rn_person <= {obs_cap}
    """)
    print(f"cohort_observation_visit rows = {spark.table(f'{TGT}.cohort_observation_visit').count()}")

    meas_select = date_clamp_select("measurement", GRAPH["measurement"])
    spark.sql(f"""
    CREATE OR REPLACE TABLE {TGT}.cohort_measurement AS
    WITH base AS (
      SELECT {meas_select}
      FROM {SRC}.measurement
      WHERE person_id IN {cohort_ids}
    ),
    ranked AS (
      SELECT *,
        row_number() OVER (PARTITION BY person_id, visit_occurrence_id ORDER BY rand({SEED})) AS rn_visit,
        row_number() OVER (PARTITION BY person_id ORDER BY rand({SEED})) AS rn_person
      FROM base
    )
    SELECT * EXCEPT (rn_visit, rn_person) FROM ranked
    WHERE rn_visit <= {PER_VISIT_KEEP} AND rn_person <= {meas_cap}
    """)
    print(f"cohort_measurement rows = {spark.table(f'{TGT}.cohort_measurement').count()}")

    for t in VALPROT_OFF_VISIT:
        sel = date_clamp_select(t, GRAPH[t])
        spark.sql(f"""
        CREATE OR REPLACE TABLE {TGT}.cohort_{t} AS
        WITH base AS (
          SELECT {sel}
          FROM {SRC}.{t}
          WHERE person_id IN {cohort_ids}
        ),
        ranked AS (
          SELECT *,
            row_number() OVER (PARTITION BY person_id, visit_occurrence_id ORDER BY rand({SEED})) AS rn_visit
          FROM base
        )
        SELECT * EXCEPT (rn_visit) FROM ranked
        WHERE rn_visit <= {PER_VISIT_KEEP}
        """)
        print(f"cohort_{t} rows = {spark.table(f'{TGT}.cohort_{t}').count()} (per-visit cap {PER_VISIT_KEEP})")

    for t in ["condition_occurrence"]:
        spark.sql(f"""
        CREATE OR REPLACE TABLE {TGT}.cohort_{t} AS
        SELECT
          {date_clamp_select(t, GRAPH[t])}
        FROM {SRC}.{t}
        WHERE person_id IN {cohort_ids}
        """)
        print(f"cohort_{t} rows = {spark.table(f'{TGT}.cohort_{t}').count()} (no cap)")

    spark.sql(f"""
    CREATE OR REPLACE TABLE {TGT}.cohort_location AS
    SELECT {date_clamp_select('location', GRAPH['location'])}
    FROM {SRC}.location
    WHERE pmod(abs(hash(location_id)), 1000000) < {max(1, int(round(COHORT['ref_pool'] / 11_467_692 * 1_000_000)))}
    """)
    spark.sql(f"""
    CREATE OR REPLACE TABLE {TGT}.cohort_care_site AS
    SELECT {date_clamp_select('care_site', GRAPH['care_site'])} FROM {SRC}.care_site
    """)
    spark.sql(f"""
    CREATE OR REPLACE TABLE {TGT}.cohort_provider AS
    SELECT {date_clamp_select('provider', GRAPH['provider'])} FROM {SRC}.provider
    """)
    for t in ["location", "care_site", "provider"]:
        print(f"cohort_{t} rows = {spark.table(f'{TGT}.cohort_{t}').count()}")
    return n_person

In [0]:
# =====================================================================================
# §2 DP-ACCOUNTED RARITY PROFILE + MEMORY GATE.
# R4b: the profiler is a DECLARED DP MECHANISM whose RELEASE artifact must be non-invertible.
#   - release mode: Laplace seeds drawn from OS entropy (secrets), NOT stored; header carries
#     NO seed and NO raw cohort_counts (both would let anyone reconstruct the noise draw).
#   - deterministic mode: cohort seed, published (header.seed=SEED) — reproducible for
#     regression/byte-diff ONLY; a real (mock=false) validate rejects it.
# =====================================================================================
def build_rarity_profile(n_person):
    release = (profiler_seed_mode == "release")
    # noise seed: entropy (unpublished) for release; cohort seed for deterministic/test.
    noise_seed = secrets.randbits(63) if release else SEED
    _eps_seqlen_family = PROFILER["epsilon_profiler"] / 2.0
    _eps_base_family = PROFILER["epsilon_profiler"] / 2.0
    _linked = [t for t in topo_order(cfg, "core") if GRAPH[t].context_fk]
    _eps_per_seqlen = _eps_seqlen_family / max(1, len(_linked))

    persons_pd = spark.table(f"{TGT}.cohort_person").select("person_id").toPandas()
    visits_pd = spark.table(f"{TGT}.cohort_visit_occurrence").select("visit_occurrence_id", "person_id").toPandas()

    def _parents_for(parent_name, parent_key):
        if parent_name == "person":
            return persons_pd.rename(columns={"person_id": parent_key}).assign(person_id=persons_pd["person_id"])
        return visits_pd[[parent_key, "person_id"]]

    prof_tables = {}
    for t in _linked:
        spec = GRAPH[t]
        pcol, pname = spec.context_fk[0], spec.context_fk[1]
        parents = _parents_for(pname, pcol)
        sel_cols = list({pcol, "person_id"})
        child = spark.table(f"{TGT}.cohort_{t}").select(*sel_cols).toPandas()
        raw = seqlen_stat(child, parents, pcol, "person_id",
                          bins=PROFILER["seqlen_bins"], parent_cap=PROFILER["seqlen_parent_cap"])
        seed_t = det_seed(noise_seed, f"{t}:seqlen")
        noised_bins = noise_bins(raw["bin_counts"], PROFILER["seqlen_parent_cap"], _eps_per_seqlen, seed_t)
        n_parents_noised = sum(noised_bins)
        frac_noised = (1.0 - (noised_bins[0] / n_parents_noised)) if n_parents_noised else 0.0
        prof_tables[t] = {"seqlen": {"n_parents": int(n_parents_noised),
                                     "frac_parent_with_child": float(frac_noised),
                                     "bin_counts": [int(x) for x in noised_bins],
                                     "bins": list(raw["bins"])}}

    death_pd = spark.table(f"{TGT}.cohort_death").select("person_id").dropna().drop_duplicates().toPandas()
    br = base_rate_stat(death_pd, persons_pd, "person_id")
    br_noised = noise_rate(br["rate"], br["n_parents"], 1, _eps_base_family, det_seed(noise_seed, "death:base_rate"))
    # R4b: publish only the NOISED rate (no n_parents_hint — that was a raw count).
    prof_tables.setdefault("death", {})["base_rate"] = {"rate": float(br_noised)}

    try:
        src_version = str(spark.sql(f"DESCRIBE HISTORY {SRC}.person LIMIT 1").collect()[0]["version"])
    except Exception:
        src_version = "unknown"
    # R4b: header carries NO seed and NO raw cohort_counts in release mode. The `seed` field
    # records the MODE label, never the actual entropy draw.
    header = profile_header(mode, {} if release else {}, RARITY_CONFIG_HASH(),
                            ("release:unpublished" if release else f"deterministic:{SEED}"),
                            PROFILER["epsilon_profiler"], src_version)
    rarity_profile = {
        "header": header,
        "privacy": {"seed_mode": profiler_seed_mode,
                    "noise_seed_published": (not release),
                    "raw_cohort_counts_published": False,
                    "note": ("release: Laplace seed from OS entropy, NOT stored; no raw counts -> "
                             "noise draw is not reconstructable. deterministic: reproducible, TEST ONLY.")},
        "epsilon_spend": {"seqlen_family": _eps_seqlen_family, "base_rate_family": _eps_base_family,
                          "per_linked_seqlen": _eps_per_seqlen, "n_linked_tables": len(_linked),
                          "note": "epsilon_profiler split /2 over seqlen + base_rate; C3 category arm dropped"},
        "tables": prof_tables,
    }
    os.makedirs(CONFIG_DIR, exist_ok=True)
    with open(f"{CONFIG_DIR}/rarity_profile.json", "w") as f:
        json.dump(rarity_profile, f, indent=2, default=str)
    print(f"wrote rarity_profile.json | config_hash={RARITY_CONFIG_HASH()} mode={mode} "
          f"seed_mode={profiler_seed_mode} linked_tables={len(_linked)}")

    rows_report = {}
    gate_fail = []
    for t in topo_order(cfg):
        tbl = f"{TGT}.cohort_{t}"
        try:
            n = spark.table(tbl).count()
        except Exception:
            n = 0
        ncols = len(src_columns(t)) if t not in ("observation_person", "observation_visit") else len(src_columns("observation"))
        est_gb = n * ncols * 24 / 1e9
        rows_report[t] = {"rows": n, "est_gb": round(est_gb, 3)}
        if n > MAX_PANDAS_ROWS or est_gb > MAX_PANDAS_GB:
            gate_fail.append(f"{t}: {n:,} rows (~{est_gb:.1f} GB)")

    report = {"mode": mode, "cohort_persons": cohort_persons, "meas_cap": meas_cap,
              "obs_cap": obs_cap, "per_visit_keep": PER_VISIT_KEEP,
              "valprot_off_visit_capped": VALPROT_OFF_VISIT,
              "fraction_bps": FRACTION_BPS, "tables": rows_report}
    with open(f"{REPORT_DIR}/cohort_{mode}.json", "w") as f:
        json.dump(report, f, indent=2)
    print(json.dumps(report, indent=2))

    if gate_fail:
        raise RuntimeError(
            "MEMORY GATE FAILED — these cohort tables exceed single-node limits "
            f"(MAX_PANDAS_ROWS={MAX_PANDAS_ROWS:,}, MAX_PANDAS_GB={MAX_PANDAS_GB}):\n  "
            + "\n  ".join(gate_fail)
            + "\n\nFIX: lower `cohort_persons` and/or `meas_cap`/`obs_cap` widgets and re-run stage=pre.")
    return rows_report

In [0]:
# =====================================================================================
# §3 MOCK GENERATION — fixture-mode test double. Returns {table: rows} for the manifest.
# =====================================================================================
def mock_generate():
    rng = np.random.default_rng(123)
    counts = {}

    def m_source_schema(name):
        base = GRAPH[name].source_table or name
        return spark.table(f"{SRC}.{base}").schema

    def m_shuffle_keys(pdf, pk, base):
        n = len(pdf)
        if n == 0:
            return pdf.copy(), {}
        new = base + rng.permutation(n) * 7 + 3
        old = pdf[pk].to_numpy()
        pdf = pdf.copy()
        pdf[pk] = new
        return pdf, dict(zip(old, new))

    _INT = (T.IntegerType, T.LongType, T.ShortType, T.ByteType)
    _FLT = (T.FloatType, T.DoubleType)

    def m_typed_sdf(name, pdf):
        sch = m_source_schema(name)
        by = {f.name: f.dataType for f in sch.fields}
        fields, out = [], pdf.copy()
        for c in pdf.columns:
            dt = by.get(c, T.StringType())
            if isinstance(dt, _INT):
                out[c] = pd.to_numeric(out[c], errors="coerce").astype("Int64")
            elif isinstance(dt, _FLT):
                out[c] = pd.to_numeric(out[c], errors="coerce").astype("float64")
            elif isinstance(dt, (T.DateType, T.TimestampType)):
                out[c] = pd.to_datetime(out[c], errors="coerce")
            else:
                dt = T.StringType()
                out[c] = out[c].astype("object").where(out[c].notna(), None)
            fields.append(T.StructField(c, dt, True))
        return spark.createDataFrame(out, schema=T.StructType(fields))

    def m_write_gen(name, pdf):
        sdf = m_typed_sdf(name, pdf)
        sdf.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{TGT}.gen_{name}")
        counts[name] = int(len(pdf))
        print(f"  gen_{name}: {len(pdf)} rows")

    def m_frame_of(name):
        all_cols = [f.name for f in spark.table(f"{TGT}.cohort_{name}").schema.fields]
        fcols = frame_columns(cfg, name, all_cols)
        pdf = spark.table(f"{TGT}.cohort_{name}").toPandas()
        return pdf[fcols], GRAPH[name]

    for t in ["location", "care_site", "provider"]:
        pdf, spec = m_frame_of(t)
        pdf, _ = m_shuffle_keys(pdf, spec.pk, PK_ID_BASE[t] + 50_000)
        m_write_gen(t, pdf)

    pdf, spec = m_frame_of("person")
    pdf, person_kmap = m_shuffle_keys(pdf, "person_id", PK_ID_BASE["person"] + 50_000)
    m_write_gen("person", pdf)

    for t in ["observation_period", "death", "condition_era", "drug_era", "dose_era", "observation_person"]:
        pdf, spec = m_frame_of(t)
        pdf["person_id"] = pdf["person_id"].map(person_kmap)
        pdf = pdf[pdf["person_id"].notna()].reset_index(drop=True)
        if spec.pk and spec.pk != "person_id":
            pdf, _ = m_shuffle_keys(pdf, spec.pk, PK_ID_BASE[t] + 50_000)
        m_write_gen(t, pdf)

    pdf, spec = m_frame_of("visit_occurrence")
    pdf["person_id"] = pdf["person_id"].map(person_kmap)
    pdf = pdf[pdf["person_id"].notna()].reset_index(drop=True)
    pdf, visit_kmap = m_shuffle_keys(pdf, "visit_occurrence_id", PK_ID_BASE["visit_occurrence"] + 50_000)
    m_write_gen("visit_occurrence", pdf)

    for t in ["measurement", "drug_exposure", "condition_occurrence",
              "procedure_occurrence", "device_exposure", "observation_visit"]:
        pdf, spec = m_frame_of(t)
        assert "person_id" not in pdf.columns, f"{t} frame must not contain person_id"
        pdf["visit_occurrence_id"] = pdf["visit_occurrence_id"].map(visit_kmap)
        pdf = pdf[pdf["visit_occurrence_id"].notna()].reset_index(drop=True)
        pdf, _ = m_shuffle_keys(pdf, spec.pk, PK_ID_BASE[t] + 50_000)
        m_write_gen(t, pdf)

    print("MOCK GENERATION COMPLETE — gen_* tables ready for assembly")
    return counts

In [0]:
# =====================================================================================
# §4 ASSEMBLE (-> STAGING) — gen_* -> stg_<table> with FULL temporal repair + PII scrub.
# =====================================================================================
def a_source_schema(table: str):
    base = (cfg.tables[table].source_table or table) if table in cfg.tables else table
    return spark.table(f"{SRC}.{base}").schema

def a_gen_pdf(name: str) -> pd.DataFrame:
    return spark.table(f"{TGT}.gen_{name}").toPandas()

_A_INT = (T.IntegerType, T.LongType, T.ShortType, T.ByteType)
_A_FLT = (T.FloatType, T.DoubleType)

def a_coerce(pdf: pd.DataFrame, sch: T.StructType) -> pd.DataFrame:
    out_pdf = pdf.copy()
    for f in sch.fields:
        c, dt = f.name, f.dataType
        s = out_pdf[c]
        if isinstance(dt, _A_INT):
            out_pdf[c] = pd.to_numeric(s, errors="coerce").astype("Int64")
        elif isinstance(dt, _A_FLT):
            out_pdf[c] = pd.to_numeric(s, errors="coerce").astype("float64")
        elif isinstance(dt, (T.DateType, T.TimestampType)):
            out_pdf[c] = pd.to_datetime(s, errors="coerce")
        elif isinstance(dt, T.BooleanType):
            out_pdf[c] = s.map(lambda x: None if pd.isna(x) else bool(x)).astype("object")
        else:
            out_pdf[c] = s.astype("object").where(s.notna(), None)
    return out_pdf

def assign_internal_ref(child, fk, parent_keyed, parent_pk, seed):
    out_pdf = child.copy()
    pool = parent_keyed[parent_pk].dropna().to_numpy()
    if len(pool) == 0 or out_pdf.empty:
        out_pdf[fk] = pd.NA
        return out_pdf
    rng = np.random.default_rng(seed)
    out_pdf[fk] = pool[rng.integers(0, len(pool), size=len(out_pdf))]
    return out_pdf

def _starts_ends(spec):
    starts = [c for c in spec.date_cols if "start" in c]
    ends = [c for c in spec.date_cols if "end" in c]
    return starts, ends

def _pairs(spec):
    """(date_col, datetime_col) pairs by naming convention: '<x>_date' + 'time' == '<x>_datetime'."""
    return [(d, d + "time") for d in spec.date_cols if (d + "time") in spec.date_cols]

def _order_all(df, spec):
    """order_pair over EVERY start/end pair (dates AND datetimes) — R2."""
    starts, ends = _starts_ends(spec)
    for s_c, e_c in zip(starts, ends):
        df = order_pair(df, s_c, e_c)
    return df


def assemble(run_id):
    """Assemble gen_* into stg_<table>; writes assemble.log + assemble_manifest.json;
    raises on failure. Returns the manifest. Finals are only touched by promote()."""
    _log = []
    def out(s):
        _log.append(str(s)); print(s)

    def write_staged(table_out, pdf, schema_table):
        sch = a_source_schema(schema_table)
        names = [f.name for f in sch.fields]
        pdf = pdf.copy()
        for c in names:
            if c not in pdf.columns:
                pdf[c] = pd.NA
        # R4a PII SCRUB: null every declared PII column of this table on write (the output
        # table name == schema_table for finals; slices carry their source table's PII too).
        pii_cols = set(PII.get(table_out, [])) | set(PII.get(schema_table, []))
        for c in pii_cols:
            if c in names:
                pdf[c] = pd.NA
        pdf = pdf[names]
        pdf = a_coerce(pdf, sch)
        nullable = T.StructType([T.StructField(f.name, f.dataType, True) for f in sch.fields])
        sdf = spark.createDataFrame(pdf, schema=nullable)
        sdf.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{TGT}.stg_{table_out}")
        scrubbed = sorted(c for c in pii_cols if c in names)
        out(f"  staged {TGT}.stg_{table_out}: {len(pdf)} rows, {len(names)} cols"
            + (f" | PII nulled: {scrubbed}" if scrubbed else ""))

    def band_guard(name, n):
        assert n < PK_BAND_SIZE, (f"{name} has {n:,} rows >= PK_BAND_SIZE {PK_BAND_SIZE:,}; "
                                  "rekeyed ids would collide across bands — raise PK_BAND_SIZE.")

    _profile_path = f"{CONFIG_DIR}/rarity_profile.json"
    _rprofile = json.load(open(_profile_path)) if os.path.exists(_profile_path) else {}
    _profile_mode = _rprofile.get("header", {}).get("cohort_mode")
    _profile_fresh = profile_is_fresh(_rprofile, RARITY_CONFIG_HASH(), _profile_mode) if _rprofile else False
    _rate_target = {t: (_rprofile.get("tables", {}).get(t, {}).get("base_rate", {}) or {}).get("rate")
                    for t in BASE_RATE_TABLES}

    try:
        # 1) REFERENCE CHAIN
        ref_frames, ref_maps = {}, {}
        _rseed = 0
        for t in REF_ORDER:
            spec = cfg.tables[t]
            df = a_gen_pdf(t); band_guard(t, len(df))
            keyed, kmap = rekey(df, spec.pk, PK_ID_BASE[t])
            for fk_col, parent in spec.internal_ref_parent.items():
                _rseed += 1
                keyed = assign_internal_ref(keyed, fk_col, ref_frames[parent],
                                            cfg.tables[parent].pk, SEED + _rseed)
            ref_frames[t] = keyed
            ref_maps[t] = kmap
        tuples = build_chain_tuples(cfg, ref_frames)
        out("reference: " + " ".join(f"{t}={len(ref_frames[t])}" for t in REF_ORDER)
            + f" tuples={len(tuples)}")

        # 2) SUBJECT
        _seed_subject = SEED + _rseed + 1
        subj_spec = cfg.tables[SUBJECT]
        subj = a_gen_pdf(SUBJECT); band_guard(SUBJECT, len(subj))
        subject_keyed, subject_map = rekey(subj, SUBJ_PK, PK_ID_BASE[SUBJECT])
        subject_keyed = assign_subject_reference(cfg, subject_keyed, tuples, _seed_subject)
        subject_keyed = clamp_valid_range(subject_keyed, subj_spec.date_cols)
        out(f"{SUBJECT}: {len(subject_keyed)} rows")

        lr = cfg.lifespan_rule or {}
        _birth_col = lr.get("birth_col")

        def _birth_floor(df, spec):
            if not _birth_col or lr.get("birth_table") != SUBJECT:
                return df
            if SUBJ_PK not in df.columns or df.empty or not spec.date_cols:
                return df
            merged = df.merge(subject_keyed[[SUBJ_PK, _birth_col]], on=SUBJ_PK, how="left",
                              suffixes=("", "__bfl"))
            bname = _birth_col if _birth_col not in df.columns else _birth_col + "__bfl"
            merged = birth_before_events(merged, bname, [c for c in spec.date_cols if c != bname])
            return merged.drop(columns=[bname])

        # 3) SUBJECT-GRAINED CHILDREN
        child_keyed = {}
        for t in SUBJECT_CHILDREN:
            spec = cfg.tables[t]
            df = a_gen_pdf(t); band_guard(t, len(df))
            df = remap_fk(df, spec.context_fk[0], subject_map, SUBJ_PK, how="inner")
            df, _ = rekey(df, spec.pk, PK_ID_BASE[t])
            df = _order_all(df, spec)
            df = _birth_floor(df, spec)
            df = _order_all(df, spec)
            df = clamp_valid_range(df, spec.date_cols)
            df = date_from_datetime(df, _pairs(spec))
            child_keyed[t] = df
            out(f"  {t}: {len(df)} rows")

        # 3b) BASE-RATE TABLES (dedup + birth floor + C2 rate calibration)
        _n_subjects = int(len(subject_keyed))
        for t in BASE_RATE_TABLES:
            spec = cfg.tables[t]
            fk_col = spec.context_fk[0]
            df = a_gen_pdf(t)
            df = remap_fk(df, fk_col, subject_map, SUBJ_PK, how="inner")
            df = df.drop_duplicates(subset=[fk_col], keep="first").reset_index(drop=True)
            target = _rate_target.get(t)
            if target is not None and _profile_fresh:
                _before = len(df)
                df = calibrate_rate(df, _n_subjects, target, SEED + 8, RARE["rate_tol"])
                out(f"  {t} calibrated: {_before} -> {len(df)} rows "
                    f"(target rate {target:.4f}, subjects {_n_subjects})")
            else:
                out(f"  {t} NOT calibrated (profile fresh={_profile_fresh}, target={target})")
            df = _birth_floor(df, spec)
            df = clamp_valid_range(df, spec.date_cols)
            child_keyed[t] = df
            out(f"  {t}: {len(df)} rows")

        # 4) HUB TABLES
        hub_keyed, hub_maps = {}, {}
        for h in HUBS:
            spec = cfg.tables[h]
            df = a_gen_pdf(h); band_guard(h, len(df))
            df = remap_fk(df, spec.context_fk[0], subject_map, SUBJ_PK, how="inner")
            df, hmap = rekey(df, spec.pk, PK_ID_BASE[h])
            if set(CHAIN_KEYS) & set(spec.ref_fks):
                df = assign_child_reference(cfg, df, h, subject_keyed, tuples, SEED + 4)
            df = _order_all(df, spec)
            df = _birth_floor(df, spec)
            df = _order_all(df, spec)
            starts, _ends = _starts_ends(spec)
            if spec.self_fk and starts:
                df = rebuild_preceding_chain(df, spec.context_fk[0], spec.pk, starts[0], spec.self_fk)
            df = clamp_valid_range(df, spec.date_cols)
            df = date_from_datetime(df, _pairs(spec))
            hub_keyed[h] = df
            hub_maps[h] = hmap
            out(f"{h}: {len(df)} rows")

        # 4b) DEATH >= LAST VISIT END: clamp base-rate table dates UP to the subject's last hub end.
        for t in BASE_RATE_TABLES:
            spec = cfg.tables[t]
            df = child_keyed[t]
            if df.empty or not HUBS:
                child_keyed[t] = date_from_datetime(df, _pairs(spec))
                continue
            last_parts = []
            for h in HUBS:
                h_spec = cfg.tables[h]
                _s, h_ends = _starts_ends(h_spec)
                end_col = h_ends[0] if h_ends else None
                if end_col and end_col in hub_keyed[h].columns:
                    last_parts.append(hub_keyed[h].groupby(SUBJ_PK)[end_col].max().rename("__last_ev"))
            if last_parts:
                last_ev = pd.concat(last_parts, axis=1).max(axis=1).rename("__last_ev") \
                    if len(last_parts) > 1 else last_parts[0]
                merged = df.merge(last_ev, left_on=spec.context_fk[0], right_index=True, how="left")
                le = pd.to_datetime(merged["__last_ev"], errors="coerce")
                n_clamped = 0
                for c in spec.date_cols:
                    if c not in merged.columns:
                        continue
                    v = pd.to_datetime(merged[c], errors="coerce")
                    bad = v.notna() & le.notna() & (v < le)
                    n_clamped = max(n_clamped, int(bad.sum()))
                    merged[c] = v.mask(bad, le)
                merged = merged.drop(columns="__last_ev")
                merged = clamp_valid_range(merged, spec.date_cols)
                merged = date_from_datetime(merged, _pairs(spec))
                child_keyed[t] = merged
                out(f"  {t}: clamped {n_clamped} rows up to last visit end")
            else:
                child_keyed[t] = date_from_datetime(df, _pairs(spec))

        # 5) HUB-GRAINED CHILDREN
        event_keyed = {}
        for h in HUBS:
            h_spec = cfg.tables[h]
            subj_col = h_spec.context_fk[0]
            for t in HUB_CHILDREN[h]:
                spec = cfg.tables[t]
                fk_col = spec.context_fk[0]
                df = a_gen_pdf(t); band_guard(t, len(df))
                df = remap_fk(df, fk_col, hub_maps[h], h_spec.pk, how="inner")
                cap_n = CAP_TABLES.get(t)
                if cap_n:
                    n_before = len(df)
                    df = cap_per_visit(df, fk_col, cap_n, SEED + 7)
                    out(f"  {t}: per-parent cap {cap_n} -> {n_before} to {len(df)} rows")
                df = derive_parent_key(df, hub_keyed[h][[h_spec.pk, subj_col]], h_spec.pk, subj_col)
                df, _ = rekey(df, spec.pk, PK_ID_BASE[t])
                if LEAF_KEY and LEAF_KEY in spec.ref_fks:
                    df = inherit_parent_col(df, hub_keyed[h], h_spec.pk, LEAF_KEY)
                df = _order_all(df, spec)
                pw = spec.parent_window
                if pw:
                    wl = df.merge(hub_keyed[h][[h_spec.pk, pw["parent_start"], pw["parent_end"]]],
                                  on=h_spec.pk, how="left")
                    df = clamp_into_window(df, spec.date_cols, wl[pw["parent_start"]], wl[pw["parent_end"]])
                df = clamp_valid_range(df, spec.date_cols)
                df = date_from_datetime(df, _pairs(spec))
                rate = NULL_FK_RATE.get(t)
                if rate:
                    df = reinject_null_fk(df, fk_col, rate, SEED + 5)
                event_keyed[t] = df
                out(f"  {t}: {len(df)} rows")

        # 6) STAGE all tables
        for t in REF_ORDER:
            write_staged(t, ref_frames[t], t)
        write_staged(SUBJECT, subject_keyed, SUBJECT)
        for t in SUBJECT_CHILDREN:
            write_staged(t, child_keyed[t], t)
        for t in BASE_RATE_TABLES:
            write_staged(t, child_keyed[t], t)
        for h in HUBS:
            write_staged(h, hub_keyed[h], h)
        _split_members = {m for ms in SPLIT_GROUPS.values() for m in ms}
        for h in HUBS:
            for t in HUB_CHILDREN[h]:
                if t not in _split_members:
                    write_staged(t, event_keyed[t], t)

        # 7) UNION split-group slices
        def _frame_of(t):
            return child_keyed[t] if t in child_keyed else event_keyed[t]
        union_rows = 0
        for gname, members in SPLIT_GROUPS.items():
            gnames = [f.name for f in a_source_schema(gname).fields]
            def _align(pdf):
                pdf = pdf.copy()
                for c in gnames:
                    if c not in pdf.columns:
                        pdf[c] = pd.NA
                return pdf[gnames]
            union = pd.concat([_align(_frame_of(t)) for t in members], ignore_index=True)
            write_staged(gname, union, gname)
            for t in members:
                write_staged(t, _frame_of(t), t)
            union_rows += len(union)

        _first_rate_table = BASE_RATE_TABLES[0] if BASE_RATE_TABLES else None
        manifest = {"status": "ok",
                    "run_id": run_id,
                    "person_rows": int(len(subject_keyed)),
                    "visit_rows": int(sum(len(hub_keyed[h]) for h in HUBS)),
                    "observation_rows": int(union_rows),
                    "reference_tuples": int(len(tuples)),
                    "capped_tables": {t: CAP_TABLES[t] for t in CAP_TABLES},
                    "death_rows": int(sum(len(child_keyed[t]) for t in BASE_RATE_TABLES)),
                    "death_target_rate": _rate_target.get(_first_rate_table),
                    "death_calibrated": bool(_first_rate_table is not None
                                             and _rate_target.get(_first_rate_table) is not None
                                             and _profile_fresh)}
        out(json.dumps(manifest, indent=2))

    except Exception as e:
        out("EXCEPTION: " + repr(e)); out(traceback.format_exc())
        manifest = {"status": "fail", "error": repr(e)}

    with open(f"{REPORT_DIR}/assemble.log", "w") as f:
        f.write("\n".join(_log) + "\n")
    with open(f"{REPORT_DIR}/assemble_manifest.json", "w") as f:
        json.dump(manifest, f, indent=2)
    if manifest.get("status") != "ok":
        raise AssertionError("assemble FAILED — see " + f"{REPORT_DIR}/assemble.log")
    return manifest

In [0]:
# =====================================================================================
# §5 VALIDATE — runs against STAGING (prefix='stg_'). Referential + semantic hard gates +
#    R4a PII-null gate + R4b release-mode gate.
# =====================================================================================
def validate(mock: bool, prefix: str = "stg_"):
    hard, soft = [], []
    def gate(name, passed, detail="", soft_gate=False):
        rec = {"gate": name, "pass": bool(passed), "detail": str(detail)}
        (soft if soft_gate else hard).append(rec)
        tag = "SOFT " if soft_gate else ("PASS " if passed else "FAIL ")
        print(tag + name + (f" — {detail}" if detail else ""))

    def count(sql):
        return spark.sql(sql).collect()[0][0]

    def _tbl(name):
        return f"{TGT}.{prefix}{name}"

    _profile_path = f"{CONFIG_DIR}/rarity_profile.json"
    _rprofile = json.load(open(_profile_path)) if os.path.exists(_profile_path) else {}

    CONTEXT_EDGES = []
    for t in CORE:
        spec = cfg.tables[t]
        if spec.context_fk:
            CONTEXT_EDGES.append((t, spec.context_fk[0], spec.context_fk[1],
                                  cfg.tables[spec.context_fk[1]].pk))
    for gname in SPLIT_GROUPS:
        CONTEXT_EDGES.append((gname, SUBJ_PK, SUBJECT, SUBJ_PK))

    REF_EDGES = []
    for t in topo_order(cfg, "reference"):
        for fk_col, parent in cfg.tables[t].internal_ref_parent.items():
            REF_EDGES.append((t, fk_col, parent, cfg.tables[parent].pk))
    for t in [SUBJECT] + HUBS:
        spec = cfg.tables[t]
        for k in CHAIN_KEYS:
            if k in spec.ref_fks:
                REF_EDGES.append((t, k, _KEY_TABLE[k], k))

    # SOFT: rarity-profile freshness
    _pmode = "fixture" if mock else "full"
    if _rprofile:
        gate("rarity_profile fresh", profile_is_fresh(_rprofile, RARITY_CONFIG_HASH(), _pmode),
             f"hash={_rprofile.get('header',{}).get('config_hash')} vs {RARITY_CONFIG_HASH()}", soft_gate=True)
    else:
        gate("rarity_profile present", False, "missing (run stage=pre)", soft_gate=True)

    # HARD (R4b): a REAL release must be built from a non-invertible (release-mode) profile.
    # In mock mode this is a soft note (fixtures may use the deterministic profile).
    _seed_mode = (_rprofile.get("privacy", {}) or {}).get("seed_mode")
    _counts_pub = (_rprofile.get("privacy", {}) or {}).get("raw_cohort_counts_published", True)
    _release_ok = (_seed_mode == "release") and (_counts_pub is False)
    gate("dp_profile_release_mode (non-invertible: entropy seed unpublished, no raw counts)",
         _release_ok, f"seed_mode={_seed_mode} raw_counts_published={_counts_pub}", soft_gate=mock)

    # HARD: orphan FKs
    orph_subs = []
    for child, fk, parent, ppk in CONTEXT_EDGES + REF_EDGES:
        orph_subs.append(f"""SELECT '{child}.{fk}->{parent}' AS edge, COUNT(*) AS orphans
          FROM {_tbl(child)} c WHERE c.`{fk}` IS NOT NULL
            AND NOT EXISTS (SELECT 1 FROM {_tbl(parent)} p WHERE p.`{ppk}` = c.`{fk}`)""")
    for r in spark.sql("\nUNION ALL\n".join(orph_subs)).collect():
        gate(f"orphan_fk {r['edge']}", r["orphans"] == 0, f"{r['orphans']} orphans")

    # HARD: reference-tuple consistency
    def _valid_source_sql(keys):
        sel, joins, covered = [], [], set()
        prev_alias = None
        for i, t in enumerate(CHAIN):
            spec = cfg.tables[t]
            alias = f"r{i}"
            own = [spec.pk] + list(spec.internal_ref_parent.keys())
            if i == 0:
                joins.append(f"{_tbl(t)} {alias}")
            else:
                joins.append(f"JOIN {_tbl(t)} {alias} ON {prev_alias}.`{spec.pk}` = {alias}.`{spec.pk}`")
            for c in own:
                if c in keys and c not in covered:
                    sel.append(f"{alias}.`{c}`")
                    covered.add(c)
            prev_alias = alias
            if covered == set(keys):
                break
        return ", ".join(sel), " ".join(joins)

    for t in [SUBJECT] + HUBS:
        spec = cfg.tables[t]
        keys = [k for k in CHAIN_KEYS if k in spec.ref_fks]
        if not keys:
            continue
        sel, src_sql = _valid_source_sql(keys)
        notnull = " AND ".join(f"x.`{k}` IS NOT NULL" for k in keys)
        match = " AND ".join(f"v.`{k}` = x.`{k}`" for k in keys)
        sql = f"""
          WITH valid AS (SELECT {sel} FROM {src_sql})
          SELECT COUNT(*) FROM {_tbl(t)} x
          WHERE {notnull}
            AND NOT EXISTS (SELECT 1 FROM valid v WHERE {match})
        """
        gate(f"ref_tuple_consistency {'person' if t == SUBJECT else ('visit' if t in HUBS else t)}",
             count(sql) == 0)

    # HARD: PK uniqueness
    pk_subs = []
    for t in topo_order(cfg) + list(SPLIT_GROUPS):
        if t in cfg.tables:
            spec = cfg.tables[t]
            pk = spec.pk if spec.pk else spec.context_fk[0]
        else:
            pk = cfg.tables[SPLIT_GROUPS[t][0]].pk
        pk_subs.append(f"""SELECT '{t}' AS tbl, '{pk}' AS pk, COUNT(*) AS tot,
          COUNT(DISTINCT `{pk}`) AS dist FROM {_tbl(t)}""")
    for r in spark.sql("\nUNION ALL\n".join(pk_subs)).collect():
        gate(f"pk_unique {r['tbl']}.{r['pk']}", r["tot"] == r["dist"], f"{r['tot']} rows / {r['dist']} distinct")

    # HARD: hub-child subject key == parent hub row's subject key
    pv_subs = []
    for h in HUBS:
        h_spec = cfg.tables[h]
        subj_col = h_spec.context_fk[0]
        for t in HUB_CHILDREN[h]:
            fk = cfg.tables[t].context_fk[0]
            pv_subs.append(f"""SELECT '{t}' AS tbl, COUNT(*) AS mism FROM {_tbl(t)} e
          JOIN {_tbl(h)} v ON e.`{fk}` = v.`{h_spec.pk}`
          WHERE e.`{fk}` IS NOT NULL AND e.`{subj_col}` <> v.`{subj_col}`""")
    if pv_subs:
        for r in spark.sql("\nUNION ALL\n".join(pv_subs)).collect():
            gate(f"person_visit_consistency {r['tbl']}", r["mism"] == 0, f"{r['mism']} mismatches")

    # HARD (R4a): every declared PII column is 100% NULL in the output.
    pii_subs = []
    for tname, cols in PII.items():
        if tname not in cfg.tables:
            continue
        present = [f.name for f in a_source_schema(tname).fields]
        for c in cols:
            if c in present:
                pii_subs.append(f"""SELECT '{tname}.{c}' AS col, COUNT(*) AS non_null
                  FROM {_tbl(tname)} WHERE `{c}` IS NOT NULL""")
    if pii_subs:
        for r in spark.sql("\nUNION ALL\n".join(pii_subs)).collect():
            gate(f"pii_null {r['col']}", r["non_null"] == 0, f"{r['non_null']} non-null (must be 0)")

    # ======================= HARD GATES (R2 — semantic temporal validity) =======================
    to_subs = []
    for t in CORE:
        spec = cfg.tables[t]
        starts = [c for c in spec.date_cols if "start" in c]
        ends = [c for c in spec.date_cols if "end" in c]
        for s_c, e_c in zip(starts, ends):
            to_subs.append(f"""SELECT '{t}.{e_c}>={s_c}' AS chk, COUNT(*) AS bad FROM {_tbl(t)}
              WHERE `{s_c}` IS NOT NULL AND `{e_c}` IS NOT NULL AND `{e_c}` < `{s_c}`""")
    if to_subs:
        for r in spark.sql("\nUNION ALL\n".join(to_subs)).collect():
            gate(f"temporal_order {r['chk']}", r["bad"] == 0, f"{r['bad']} rows end<start")

    dm_subs = []
    for t in CORE:
        spec = cfg.tables[t]
        for d_c in spec.date_cols:
            dt_c = d_c + "time"
            if dt_c in spec.date_cols:
                dm_subs.append(f"""SELECT '{t}.{d_c}' AS chk, COUNT(*) AS bad FROM {_tbl(t)}
                  WHERE `{d_c}` IS NOT NULL AND `{dt_c}` IS NOT NULL
                    AND CAST(`{d_c}` AS DATE) <> CAST(`{dt_c}` AS DATE)""")
    if dm_subs:
        for r in spark.sql("\nUNION ALL\n".join(dm_subs)).collect():
            gate(f"date_matches_datetime {r['chk']}", r["bad"] == 0, f"{r['bad']} mismatches")

    lr = cfg.lifespan_rule or {}
    if lr.get("birth_col") and lr.get("birth_table") == SUBJECT:
        bf_subs = []
        for t in CORE:
            spec = cfg.tables[t]
            if t == SUBJECT or not spec.date_cols:
                continue
            conds = " OR ".join(f"(e.`{c}` IS NOT NULL AND e.`{c}` < p.`{lr['birth_col']}`)"
                                for c in spec.date_cols)
            bf_subs.append(f"""SELECT '{t}' AS tbl, COUNT(*) AS bad FROM {_tbl(t)} e
              JOIN {_tbl(SUBJECT)} p ON e.`{SUBJ_PK}` = p.`{SUBJ_PK}`
              WHERE p.`{lr['birth_col']}` IS NOT NULL AND ({conds})""")
        for r in spark.sql("\nUNION ALL\n".join(bf_subs)).collect():
            gate(f"birth_floor {r['tbl']}", r["bad"] == 0, f"{r['bad']} rows before birth")

    if BASE_RATE_TABLES and HUBS and lr.get("death_table") in BASE_RATE_TABLES:
        dt_name = lr["death_table"]
        h = HUBS[0]
        h_spec = cfg.tables[h]
        _s, h_ends = _starts_ends(h_spec)
        if h_ends:
            end_col = h_ends[0]
            death_date_col = [c for c in cfg.tables[dt_name].date_cols if not c.endswith("time")]
            dcol = death_date_col[0] if death_date_col else cfg.tables[dt_name].date_cols[0]
            sql = f"""
              SELECT COUNT(*) FROM {_tbl(dt_name)} d
              JOIN (SELECT `{SUBJ_PK}`, MAX(`{end_col}`) AS mx FROM {_tbl(h)} GROUP BY `{SUBJ_PK}`) v
                ON d.`{SUBJ_PK}` = v.`{SUBJ_PK}`
              WHERE d.`{dcol}` IS NOT NULL AND v.mx IS NOT NULL AND d.`{dcol}` < v.mx
            """
            gate(f"death_after_last_visit {dt_name}.{dcol}", count(sql) == 0)

    for t, cap_n in CAP_TABLES.items():
        fk_col = cfg.tables[t].context_fk[0]
        sql = f"""SELECT COALESCE(MAX(n), 0) FROM (
          SELECT COUNT(*) AS n FROM {_tbl(t)} WHERE `{fk_col}` IS NOT NULL GROUP BY `{fk_col}`)"""
        mx = count(sql)
        gate(f"cap_enforced {t} (per_parent<={cap_n})", mx <= cap_n, f"max per parent = {mx}")

    # HARD/SOFT: empty-table expected-count
    ec_subs = []
    for t in CORE:
        spec = cfg.tables[t]
        sl = (_rprofile.get("tables", {}).get(t, {}) or {}).get("seqlen", {})
        frac = sl.get("frac_parent_with_child")
        if not spec.context_fk or frac is None:
            continue
        ec_subs.append(f"""SELECT '{t}' AS tbl, CAST({frac} AS DOUBLE) AS frac,
          (SELECT COUNT(*) FROM {_tbl(spec.context_fk[1])}) AS n_parent,
          (SELECT COUNT(*) FROM {_tbl(t)}) AS actual""")
    if ec_subs:
        for r in spark.sql("\nUNION ALL\n".join(ec_subs)).collect():
            expected = r["frac"] * r["n_parent"]
            if expected >= RARE["min_expected"]:
                gate(f"nonempty {r['tbl']} (expected~{expected:.0f})", r["actual"] > 0, f"actual={r['actual']}")
            else:
                gate(f"nonempty {r['tbl']} (expected~{expected:.1f}<min)", True,
                     f"actual={r['actual']}", soft_gate=True)

    # SOFT: configured NULL-rate tolerance
    def null_rate(table, col):
        tot = count(f"SELECT COUNT(*) FROM {_tbl(table)}")
        if tot == 0:
            return 0.0
        nul = count(f"SELECT COUNT(*) FROM {_tbl(table)} WHERE `{col}` IS NULL")
        return nul / tot
    _nc = (cfg.extra.get("validate") or {}).get("null_rate_check")
    if _nc:
        r = null_rate(_nc["table"], _nc["col"])
        gate(_nc["name"], _nc["lo"] <= r <= _nc["hi"], f"{r:.4f}", soft_gate=True)

    # HARD: DP boundary (soft only in mock mode)
    try:
        with open(f"{REPORT_DIR}/dp_audit.json") as f:
            dp_audit = json.load(f)
        composed = dp_audit.get("composed_epsilon_naive_sum")
        dp_ok = bool(dp_audit.get("enabled")) and dp_audit.get("n_dp_tables") == len(CORE)
        detail = (f"n_dp={dp_audit.get('n_dp_tables')}/{len(CORE)} "
                  f"valprot_on={dp_audit.get('n_value_protection_tables')} "
                  f"composed_eps_naive={composed} (incl profiler {dp_audit.get('profiler_epsilon')})")
        gate("dp_boundary DP on every core table (honest composed eps reported)", dp_ok, detail)
    except FileNotFoundError:
        gate("dp_boundary", mock, "dp_audit.json missing (mock run — tolerated)" if mock
             else "dp_audit.json missing — run pipeline_train", soft_gate=mock)

    # FIDELITY (informational)
    _v = cfg.extra.get("validate") or {}

    def ratio(table):
        p = count(f"SELECT COUNT(DISTINCT `{SUBJ_PK}`) FROM {_tbl(SUBJECT)}")
        n = count(f"SELECT COUNT(*) FROM {_tbl(table)}")
        return (n / p) if p else 0.0

    _synth = {}
    for label, table in (_v.get("fidelity_ratios") or {}).items():
        nd = 4 if label.endswith("_fraction") else 2
        _synth[label] = round(ratio(table), nd)

    _first_rate = BASE_RATE_TABLES[0] if BASE_RATE_TABLES else None
    fidelity = {
        "synthetic": _synth,
        "source_reference": _v.get("source_reference") or {},
        "death_target_rate": ((_rprofile.get("tables", {}).get(_first_rate, {}) or {})
                              .get("base_rate", {}) or {}).get("rate") if _first_rate else None,
    }

    _cc = _v.get("cooccurrence")
    if _cc:
        try:
            co = spark.sql(f"""
              WITH top_parent AS (SELECT `{_cc['parent_concept']}` AS pc FROM {_tbl(_cc['parent'])}
                GROUP BY `{_cc['parent_concept']}` ORDER BY COUNT(*) DESC LIMIT 1)
              SELECT m.`{_cc['child_concept']}` AS concept, COUNT(*) AS n
              FROM {_tbl(_cc['child'])} m
              JOIN {_tbl(_cc['parent'])} v
                ON m.`{cfg.tables[_cc['child']].context_fk[0]}` = v.`{cfg.tables[_cc['parent']].pk}`
              WHERE v.`{_cc['parent_concept']}` = (SELECT pc FROM top_parent)
              GROUP BY m.`{_cc['child_concept']}` ORDER BY n DESC LIMIT 5""").collect()
            fidelity[_cc["key"]] = [{_cc["child_concept"]: r["concept"], "n": r["n"]} for r in co]
        except Exception as e:
            fidelity["cooccurrence_error"] = str(e)

    n_hard_fail = sum(1 for r in hard if not r["pass"])
    n_soft_warn = sum(1 for r in soft if not r["pass"])
    report = {"mock": mock, "validated_prefix": prefix, "hard_gates": hard, "soft_checks": soft,
              "n_hard_fail": n_hard_fail, "n_soft_warn": n_soft_warn, "fidelity": fidelity}
    with open(f"{REPORT_DIR}/validation.json", "w") as f:
        json.dump(report, f, indent=2, default=str)
    print(json.dumps(report, indent=2, default=str))

    summary = (f"PASS: all {len(hard)} hard gates ({n_soft_warn} soft warnings)"
               if n_hard_fail == 0 else f"FAIL: {n_hard_fail} hard gate(s) failed")
    if n_hard_fail:
        raise AssertionError(summary + " — see " + f"{REPORT_DIR}/validation.json")
    return summary

In [0]:
# =====================================================================================
# §6 PROMOTE — swap validated staging tables into the finals, tagged with run_id;
#    append the run's epsilon to the cumulative ledger (R4c).
# =====================================================================================
def _append_epsilon_ledger(run_id):
    """Append this run's composed epsilon to epsilon_ledger.json; report the cumulative
    naive-sum across PROMOTED runs. Dedup by run_id (re-promote of the same run is idempotent).
    NOTE: only promotions are ledgered — training that never promotes is not counted (a
    conservative under-count is avoided by only ledgering released data)."""
    try:
        with open(f"{REPORT_DIR}/dp_audit.json") as f:
            dp = json.load(f)
        eps = dp.get("composed_epsilon_naive_sum")
    except Exception:
        dp, eps = {}, None
    prof = {}
    try:
        prof = json.load(open(f"{CONFIG_DIR}/rarity_profile.json")).get("privacy", {})
    except Exception:
        pass
    try:
        ledger = json.load(open(LEDGER_PATH))
    except Exception:
        ledger = {"delta": cfg.dp["delta"], "runs": []}
    ledger["runs"] = [r for r in ledger.get("runs", []) if r.get("run_id") != run_id]
    ledger["runs"].append({"run_id": run_id, "promoted_utc": _utcnow(),
                           "composed_epsilon_naive_sum": eps,
                           "profiler_seed_mode": prof.get("seed_mode"),
                           "output_persons": (read_manifest() or {}).get("generation", {}).get("output_persons")})
    ledger["cumulative_epsilon_naive_sum"] = sum(r["composed_epsilon_naive_sum"] or 0 for r in ledger["runs"])
    ledger["n_promoted_runs"] = len(ledger["runs"])
    with open(LEDGER_PATH, "w") as f:
        json.dump(ledger, f, indent=2, default=str)
    print(f"epsilon ledger: this run={eps}, cumulative naive-sum={ledger['cumulative_epsilon_naive_sum']} "
          f"over {ledger['n_promoted_runs']} promoted run(s), delta={ledger['delta']}")
    return ledger

def promote(run_id):
    for t in ALL_FINAL_TABLES:
        spark.sql(f"CREATE OR REPLACE TABLE {TGT}.{t} "
                  f"TBLPROPERTIES ('synth_run_id' = '{run_id}') "
                  f"AS SELECT * FROM {TGT}.stg_{t}")
    for t in ALL_FINAL_TABLES:
        spark.sql(f"DROP TABLE IF EXISTS {TGT}.stg_{t}")
    print(f"PROMOTED {len(ALL_FINAL_TABLES)} tables (synth_run_id={run_id}); staging dropped")
    _append_epsilon_ledger(run_id)

In [0]:
# =====================================================================================
# DISPATCH — pre (manifest created) / post (manifest-verified -> stage -> validate -> promote).
# =====================================================================================
result = {"mode": mode, "stage": stage}

if stage in ("auto", "pre"):
    provision()
    n_person = build_cohort()
    build_rarity_profile(n_person)
    run_id = uuid.uuid4().hex[:12]
    write_manifest({"run_id": run_id, "created_utc": _utcnow(), "mode": mode,
                    "cohort": {"persons": cohort_persons, "meas_cap": meas_cap,
                               "obs_cap": obs_cap, "per_visit_keep": PER_VISIT_KEEP,
                               "fraction_bps": FRACTION_BPS, "seed": SEED},
                    "profile_config_hash": RARITY_CONFIG_HASH(),
                    "profiler_seed_mode": profiler_seed_mode,
                    "stage": "cohort_ready"})
    print(f"run manifest created: run_id={run_id}")
    result["run_id"] = run_id
    result["cohort_person_rows"] = int(n_person)

def _post_manifest():
    """Verify the run manifest + gen_* coherence before assembly. Returns run_id."""
    m = read_manifest()
    if not m or "generation" not in m:
        if force_ignore_manifest:
            rid = (m or {}).get("run_id") or ("legacy_" + uuid.uuid4().hex[:8])
            print(f"WARNING force_ignore_manifest=true — proceeding without a generation record "
                  f"(run_id={rid}). Provenance of gen_* is NOT verified.")
            return rid
        raise RuntimeError(
            "run_manifest.json has no generation record — pipeline_train has not recorded a "
            "generation for this run (or the manifest is missing). Run pipeline_train, or set "
            "force_ignore_manifest=true to assemble unverified legacy gen_* tables.")
    mism = []
    for t, n in m["generation"].get("tables", {}).items():
        try:
            live = spark.table(f"{TGT}.gen_{t}").count()
        except Exception:
            live = -1
        if int(live) != int(n):
            mism.append(f"gen_{t}: manifest={n} live={live}")
    if mism:
        raise RuntimeError(
            "STALE/MIXED GENERATION — live gen_* tables do not match the run manifest "
            f"(run_id={m['run_id']}):\n  " + "\n  ".join(mism)
            + "\nRe-run pipeline_train (or investigate) before assembling.")
    print(f"gen_* verified against manifest (run_id={m['run_id']}, "
          f"{len(m['generation'].get('tables', {}))} tables)")
    return m["run_id"]

if mode == "fixture" and stage in ("auto", "post"):
    print("FIXTURE mode: using MOCK generation (no mostlyai) to produce gen_* tables")
    counts = mock_generate()
    m = read_manifest() or {"run_id": "fixture_" + uuid.uuid4().hex[:8], "mode": mode}
    m["generation"] = {"recorded_utc": _utcnow(), "kind": "mock", "tables": counts}
    m["stage"] = "generated"
    write_manifest(m)
    manifest = assemble(m["run_id"])
    summary = validate(mock=True)
    promote(m["run_id"])
    m["stage"] = "promoted"
    m["promoted_utc"] = _utcnow()
    write_manifest(m)
    result.update({"status": "fixture_complete", "run_id": m["run_id"],
                   "assemble": manifest, "validate": summary})
    dbutils.notebook.exit(json.dumps(result, default=str))

if stage in ("auto", "post") and mode == "full":
    m = read_manifest()
    if stage == "auto" and (not m or "generation" not in m) and not force_ignore_manifest:
        msg = (
            "\n" + "=" * 78 +
            "\nPAUSED — RUN pipeline_train ON THE ML CLUSTER\n" + "=" * 78 +
            f"\nCohort tables are ready in {TGT}.cohort_*.\n"
            "On the ML cluster (mostlyai + GPU), run synth_omop/pipeline_train — it trains the\n"
            f"generators, writes {TGT}.gen_*, and records the generation in the run manifest.\n"
            "Then re-run this pipeline with stage=post to assemble -> validate -> promote.\n"
            + "=" * 78)
        print(msg)
        result.update({"status": "awaiting_pipeline_train",
                       "run_id": (m or {}).get("run_id"),
                       "reference_zip": os.path.exists(GEN_REFERENCE),
                       "core_zip": os.path.exists(GEN_CORE)})
        dbutils.notebook.exit(json.dumps(result, default=str))
    run_id = _post_manifest()
    manifest = assemble(run_id)
    summary = validate(mock=False)
    promote(run_id)
    m = read_manifest() or {"run_id": run_id}
    m["stage"] = "promoted"
    m["promoted_utc"] = _utcnow()
    write_manifest(m)
    result.update({"status": "complete", "run_id": run_id,
                   "assemble": manifest, "validate": summary})
    dbutils.notebook.exit(json.dumps(result, default=str))

result.setdefault("status", "pre_complete")
result["next"] = "run pipeline_train on the ML cluster, then re-run with stage=post"
dbutils.notebook.exit(json.dumps(result, default=str))